openssl tp

The first two parts have saved outputs. The image and hybrid parts need your lab files; edit the filenames before running them.

mission 1

In [1]:
# Mission 1: RSA arithmetic with the lab's deliberately tiny parameters.
p, q, e, ciphertext = 5, 11, 3, 23
n = p * q
phi = (p - 1) * (q - 1)
d = pow(e, -1, phi)
message = pow(ciphertext, d, n)
print("n =", n)
print("phi(n) =", phi)
print("Public key (n, e) =", (n, e))
print("Private key (n, d) =", (n, d))
print("Recovered password =", message)
print("Re-encrypted ciphertext =", pow(message, e, n))
assert pow(message, e, n) == ciphertext


n = 55
phi(n) = 40
Public key (n, e) = (55, 3)
Private key (n, d) = (55, 27)
Recovered password = 12
Re-encrypted ciphertext = 23


small key

In [2]:
from pathlib import Path

Path("otherSmall_pub.pem").write_text('-----BEGIN PUBLIC KEY-----\nMCIwDQYJKoZIhvcNAQEBBQADEQAwDgIHAtcS9BRJHQIDAQAB\n-----END PUBLIC KEY-----\n')


In [3]:
# Mission 3: trial division, restricted to the tiny classroom public key.
import argparse
import math
import subprocess
import time

parser = argparse.ArgumentParser()
parser.add_argument("key", nargs="?", default="otherSmall_pub.pem")
parser.add_argument("--seconds", type=float, default=5.0)
args = parser.parse_args([])
der = subprocess.run(["openssl", "pkey", "-pubin", "-in", args.key,
                      "-pubout", "-outform", "DER"], check=True, capture_output=True).stdout

def tlv(data, pos=0):
    tag, size = data[pos], data[pos+1]
    pos += 2
    if size & 128:
        count = size & 127
        size = int.from_bytes(data[pos:pos+count], "big")
        pos += count
    end = pos + size
    if end > len(data):
        raise ValueError("Truncated DER")
    return tag, data[pos:end], end

_, outer, _ = tlv(der)
_, _, pos = tlv(outer)  # Skip the algorithm identifier.
tag, bit_string, _ = tlv(outer, pos)
if tag != 3 or bit_string[0] != 0:
    raise SystemExit("Unexpected public key encoding")
_, rsa_sequence, _ = tlv(bit_string[1:])
_, n_bytes, pos = tlv(rsa_sequence)
_, e_bytes, _ = tlv(rsa_sequence, pos)
n, e = int.from_bytes(n_bytes, "big"), int.from_bytes(e_bytes, "big")
print("n =", n, "(bits:", n.bit_length(), ")")
print("e =", e)
start = time.monotonic()
limit = math.isqrt(n)
factor = 2 if n % 2 == 0 else None
candidate = 3
while factor is None and candidate <= limit:
    if n % candidate == 0:
        factor = candidate
        break
    if candidate % 20001 == 0 and time.monotonic() - start > args.seconds:
        raise SystemExit("Time limit reached; no factor found by trial division.")
    candidate += 2
if factor is None:
    raise SystemExit("No nontrivial factor found")
p, q = factor, n // factor
phi = (p - 1) * (q - 1)
d = pow(e, -1, phi)
print("p =", p)
print("q =", q)
print("phi(n) =", phi)
print("d =", d)
assert p * q == n and e * d % phi == 1


n = 799426357774621 (bits: 50 )
e = 65537
p = 26489357
q = 30179153
phi(n) = 799426301106112
d = 562161229729409


image blocks — needs the encrypted image

In [ ]:
# Mission 2: show repeated AES blocks without decrypting them.
import argparse
from collections import Counter
from pathlib import Path
from PIL import Image

parser = argparse.ArgumentParser()
parser.add_argument("input")
parser.add_argument("output")
args = parser.parse_args(["encrypted.bin", "blocks.png"])
width, height = 432, 475
pixel_count = width * height
# PKCS#7 adds a complete padding block when input is block-aligned.
expected_cipher_size = (pixel_count // 16 + 1) * 16
data = Path(args.input).read_bytes()
if data[:8] != b"Salted__":
    raise SystemExit("Expected an OpenSSL Salted__ file.")
# Infer the salt length from the known raw image size and padding.
header_size = len(data) - expected_cipher_size
if header_size not in (16, 24):
    raise SystemExit("Unexpected file size: expected an 8- or 16-byte salt.")
print("Signature:", data[:8])
print("Salt:", data[8:header_size].hex())
print("Excluded header bytes:", header_size)
ciphertext = data[header_size:]
all_blocks = [ciphertext[i:i+16] for i in range(0, len(ciphertext), 16)]
# Exclude the final full padding block from both image and repetition counts.
blocks = all_blocks[:pixel_count // 16]
counts = Counter(blocks)
shades = {block: (index * 73) % 256 for index, block in enumerate(counts)}
pixels = b"".join(bytes([shades[block]]) * 16 for block in blocks)
Image.frombytes("L", (width, height), pixels).save(args.output)
print("Image blocks:", len(blocks))
print("Unique image blocks:", len(counts))
print("Distinct blocks occurring more than once:", sum(n > 1 for n in counts.values()))
print("Repeated occurrences beyond the first:", sum(n - 1 for n in counts.values()))
print("Saved:", args.output)
# There are only 256 shades; distinct blocks can share a shade.


raw image — needs recovered.raw

In [ ]:
# Mission 2, Challenge 3.
import argparse
from pathlib import Path
from PIL import Image

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default="recovered.raw")
parser.add_argument("output", nargs="?", default="recovered.png")
args = parser.parse_args(["recovered.raw", "recovered.png"])
data = Path(args.input).read_bytes()
if len(data) != 432 * 475:
    raise SystemExit(f"Expected 205200 bytes, received {len(data)}.")
Image.frombytes("L", (432, 475), data).save(args.output)
print("Saved:", args.output)


hybrid — needs the public key and message

For decryption, change the argument list to `["decrypt", "private.pem", "package", "recovered.txt"]`.

In [ ]:
# Mission 3, 5.14-5.15: AES-256-CBC + RSA-OAEP-SHA256.
# Classroom scheme: CBC does not authenticate the message or the IV.
import argparse
import secrets
import subprocess
import tempfile
from pathlib import Path

OAEP = ["-pkeyopt", "rsa_padding_mode:oaep",
        "-pkeyopt", "rsa_oaep_md:sha256",
        "-pkeyopt", "rsa_mgf1_md:sha256"]

def run(*args):
    subprocess.run(["openssl", *args], check=True)

parser = argparse.ArgumentParser()
sub = parser.add_subparsers(dest="mode", required=True)
send = sub.add_parser("encrypt")
send.add_argument("public_key")
send.add_argument("plaintext")
send.add_argument("package")
receive = sub.add_parser("decrypt")
receive.add_argument("private_key")
receive.add_argument("package")
receive.add_argument("output")
args = parser.parse_args(["encrypt", "public.pem", "message.txt", "package"])
package = Path(args.package)
with tempfile.TemporaryDirectory() as temporary:
    key_file = Path(temporary) / "aes.key"
    if args.mode == "encrypt":
        # Refuse to overwrite a previously created package.
        package.mkdir(parents=True, exist_ok=False)
        key, iv = secrets.token_bytes(32), secrets.token_bytes(16)
        key_file.write_bytes(key)
        run("pkeyutl", "-encrypt", "-pubin", "-inkey", args.public_key,
            "-in", str(key_file), "-out", str(package / "aes_key.enc"), *OAEP)
        run("enc", "-aes-256-cbc", "-nosalt", "-K", key.hex(), "-iv", iv.hex(),
            "-in", args.plaintext, "-out", str(package / "message.enc"))
        (package / "iv.bin").write_bytes(iv)
        print("Send the package directory; it contains the ciphertext, wrapped key and IV.")
    else:
        run("pkeyutl", "-decrypt", "-inkey", args.private_key,
            "-in", str(package / "aes_key.enc"), "-out", str(key_file), *OAEP)
        key, iv = key_file.read_bytes(), (package / "iv.bin").read_bytes()
        if len(key) != 32 or len(iv) != 16:
            raise SystemExit("Invalid AES key or IV length")
        run("enc", "-d", "-aes-256-cbc", "-nosalt", "-K", key.hex(), "-iv", iv.hex(),
            "-in", str(package / "message.enc"), "-out", args.output)
        print("Recovered:", args.output)
